# Visualise ablation results

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import os
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from src.constants import LOCAL_RESULTS_DIR, DATASPLITS_MKQA, DATASPLITS_GMMLU

In [ ]:
# --- Colors ---
WORSE = "#772244"   # red - worse model
BETTER = "#117733"  # green - better model
LINE  = "#332288"   # blue

def _importance_legend(metric_name: str, better_when_positive: bool):
    m = metric_name
    if better_when_positive:
        # Δ>0 means better (your Brier/ECE plotting convention)
        better_patch = mpatches.Patch(color=BETTER, alpha=0.55, label=f"Δ{m} > 0 (better model)")
        worse_patch  = mpatches.Patch(color=WORSE,  alpha=0.55, label=f"Δ{m} < 0 (worse model)")
    else:
        # Δ>0 means worse (AUROC/AUPR)
        worse_patch  = mpatches.Patch(color=WORSE,  alpha=0.55, label=f"Δ{m} > 0 (worse model)")
        better_patch = mpatches.Patch(color=BETTER, alpha=0.55, label=f"Δ{m} < 0 (better model)")
    return [worse_patch, better_patch]

def _align_zero_and_fit(ax_left, ax_right, line_values, pad_frac=0.08):
    y0_left_min, y0_left_max = ax_left.get_ylim()
    zero_pos = -y0_left_min / max(y0_left_max - y0_left_min, 1e-12)

    finite_vals = np.asarray(line_values)[np.isfinite(line_values)]
    if finite_vals.size == 0:
        yr_min, yr_max = ax_right.get_ylim()
        rng = max(yr_max - yr_min, 1.0)
    else:
        data_min, data_max = finite_vals.min(), finite_vals.max()
        span = data_max - data_min
        margin = pad_frac * (span if span > 0 else (abs(data_max) + abs(data_min) + 1e-9))
        want_min = data_min - margin
        want_max = data_max + margin
        r1 = (-want_min) / max(zero_pos, 1e-12)
        r2 = ( want_max) / max(1 - zero_pos, 1e-12)
        rng = max(r1, r2, 1.0)

    ymin = -zero_pos * rng
    ymax = (1 - zero_pos) * rng
    ax_right.set_ylim(ymin, ymax)

def _flip_if_brier_ece(metric_name: str, values: np.ndarray) -> np.ndarray:
    """
    Your CSV has Δ_abl: AUROC/AUPR = base - masked, Brier/ECE = masked - base.
    You want to plot Δ_plot = base - masked for *Brier/ECE* too.
    So for Brier/ECE we flip sign at plot time.
    """
    if metric_name.lower() in ("brier", "ece"):
        return -values
    return values

def _better_when_positive(metric_name: str) -> bool:
    """
    After the transform above, Δ>0 means:
      - AUROC/AUPR: worse (unchanged)
      - Brier/ECE : better (by design)
    """
    return metric_name.lower() in ("brier", "ece")

def plot_one(dfi: pd.DataFrame, label: str, fname: str, metric_col: str, metric_name: str, ylabel: str, ylim: tuple, yticks: np.ndarray, *, alpha_mean: np.ndarray | None = None, output_dir: str = ".",
             num_layers=32):
    
    # figure out L
    L = int(num_layers) if num_layers is not None else int(dfi["layer"].max()) + 1

    # aggregate
    mean_by_layer = dfi.groupby("layer", dropna=False)[metric_col].mean().reindex(range(L))
    std_by_layer  = dfi.groupby("layer", dropna=False)[metric_col].std().reindex(range(L))

    layers   = np.arange(L) + 1
    mean_imp_raw = mean_by_layer.values
    std_imp  = np.nan_to_num(std_by_layer.values, nan=0.0)

    # ---- apply your sign convention for Brier/ECE ----
    mean_imp = _flip_if_brier_ece(metric_name, mean_imp_raw)
    better_pos = _better_when_positive(metric_name)

    # ---- colors & legend ----
    colors = np.where(mean_imp > 0, BETTER if better_pos else WORSE,
                               WORSE if better_pos else BETTER)

    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax1.bar(layers, mean_imp, yerr=std_imp, align="center", alpha=0.55, capsize=3, color=colors, ecolor="black", error_kw=dict(alpha=1, lw=1))
    ax1.axhline(0.0, linestyle="--", linewidth=1, color="black")
    ax1.set_xlabel("Layer", fontsize=20)
    ax1.set_ylabel(ylabel, fontsize=20)
    ax1.set_title(f"{label}", fontsize=24)
    ax1.set_ylim(*ylim)
    ax1.set_yticks(yticks)
    ax1.grid(True, axis="both", linestyle="--", alpha=0.5)
    ax1.set_xticks(np.arange(1, L + 1, 2))
    ax1.tick_params(axis="both", labelsize=16)

    # optional α line
    line_handle = None
    if alpha_mean is not None and np.all(np.isfinite(alpha_mean)) and len(alpha_mean) == L:
        ax2 = ax1.twinx()
        (line_handle,) = ax2.plot(layers, alpha_mean, marker="o", linewidth=1.8, color=LINE, alpha=0.8, label=r"$\alpha_\ell$")
        ax2.set_ylabel("Layer importance", fontsize=20)
        ax2.set_xlim(0.5, L + 0.5)
        _align_zero_and_fit(ax1, ax2, alpha_mean, pad_frac=0.08)
        
        # set the range of alpha
        ymin_cur, ymax_cur = ax2.get_ylim()
        zero_pos = -ymin_cur / max(ymax_cur - ymin_cur, 1e-12)  # fraction below zero

        pad_frac = 0.10               # <-- add a small top padding (10%)
        target_max = 0.1 * (1 + pad_frac)
        # solve for bottom so zero stays aligned
        target_min = - (zero_pos / max(1 - zero_pos, 1e-12)) * target_max
        ax2.set_ylim(target_min, target_max)
        
        ax2.patch.set_visible(False)
        ax2.tick_params(axis="both", labelsize=16)

    handles = _importance_legend(metric_name, better_when_positive=better_pos)
    if line_handle is not None:
        handles.append(line_handle)
    leg = ax1.legend(handles=handles, loc="lower right", frameon=True, fontsize=15)
    # increase font size just for αₗ
    for text in leg.get_texts():
        if r"$\alpha_\ell$" in text.get_text():
            text.set_fontsize(19)   # set bigger font just for αₗ

    os.makedirs(output_dir, exist_ok=True)
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, f"{fname}.png"), dpi=300, bbox_inches="tight")
    # plt.show()
    plt.close(fig)


## Weight ablation

In [ ]:
# -------------------- CONFIG --------------------
PROBE = "SoftmaxLayerProbe"
L1_W = 0
USE_CALIBRATED = False
SEEDS = [42, 17, 3, 59, 884, 445, 369, 169, 76, 905]
train_lang = "fr"

# DATASET = "mkqa"
DATASET = "global_mmlu"
# LLM   = "llama_3.1_8B"
LLM     = "qwen3_8B"

EXCLUDE_TIME_SENSITIVE = True if DATASET == "mkqa" else False
DATASPLITS   = DATASPLITS_MKQA if DATASET == "mkqa" else DATASPLITS_GMMLU
NUM_LAYERS   = 32 if LLM == "llama_3.1_8B" else 36
HIDDEN_DIM   = 4096
TRAIN_BATCH  = 32
STRATIFIED   = True
K_WINDOW     = 3

RESULTS_ROOT = Path(LOCAL_RESULTS_DIR) / DATASET / LLM

# ---- hyperparams that identify the trained probe ----
# MKQA - LLAMA 3.1 8 B
# QUERY = True
# LR = 0.0008
# EPOCHS = 139 
# L2_V = 0.286102950514828

# QUERY = False
# LR = 0.0005
# EPOCHS = 210
# L2_V = 0.27276092363216636

# ---- MKQA - QWEN 3 8B
# QUERY = True
# LR = 0.0009
# EPOCHS = 98
# L2_V = 0.29527869618547636

# QUERY = False
# LR = 0.0009
# EPOCHS = 111
# L2_V = 0.22252863623823427

# ---- GLOBAL MMLU - LLAMA 3.1 8B
# QUERY = True
# LR = 0.0011
# EPOCHS = 94 
# L2_V = 0.28258845606106203

# QUERY = False
# LR = 0.0020
# EPOCHS = 60
# L2_V = 0.2557367492257226

# ---- GLOBAL MMLU - QWEN 3 8B
# QUERY = True
# LR = 0.001
# EPOCHS = 103
# L2_V = 0.2686538578969079

QUERY = False
LR = 0.001
EPOCHS = 165
L2_V = 0.2115332075762067

In [ ]:
# -------------------- PATH BUILDING --------------------
probe_type = "query" if QUERY else "answer"
if QUERY:
    probe_folder = "probe_query_without_time_sensitive" if EXCLUDE_TIME_SENSITIVE else "probe_query"
else:
    probe_folder = "probe_answer_without_time_sensitive" if EXCLUDE_TIME_SENSITIVE else "probe_answer"

per_layer_csv = (
    Path(LOCAL_RESULTS_DIR)
    / f"{DATASET}/{LLM}/{probe_folder}"
    / f"SoftmaxLayerProbe_layer_scalers/train_{train_lang}"
    / f"lr_{LR}_l1_{L1_W}_l2_{L2_V}_epochs_{EPOCHS}_readout_ablation_k{K_WINDOW}_per_layer_multiseed.csv"
)
models_root = (
    Path(LOCAL_RESULTS_DIR)
    / f"{DATASET}/{LLM}/{probe_folder}"
    / f"SoftmaxLayerProbe_layer_scalers/train_{train_lang}"
)

In [ ]:
# -------------------- α: compute once from EXACT seed paths --------------------
alpha_path = models_root / f"alpha_mean_lr_{LR}_l1_{L1_W}_l2_{L2_V}_epochs_{EPOCHS}.npy"

def load_alpha_mean_exact(
    models_root: Path, seeds, num_layers: int,
    lr: float, l1_w: float, l2_v: float, epochs: int
) -> np.ndarray:
    alphas = []
    missing = []
    for seed in seeds:
        ckpt = (
            models_root
            / f"batch_32_lr_{lr}_l1_{l1_w}_l2_{l2_v}_epochs_{epochs}_seed_{seed}"  # FIX: added batch_32_ prefix
            / "probe_state.pt"
        )
        if not ckpt.exists():
            missing.append(seed)
            continue
        st = torch.load(ckpt, map_location="cpu")
        w = st["w"].reshape(-1)
        if w.numel() != num_layers:
            raise ValueError(f"[seed {seed}] expected {num_layers} layers, got {w.numel()}")
        alphas.append(F.softmax(w, dim=-1).numpy())
    if missing:
        print(f"[warn] missing checkpoints for seeds: {missing}")
    if not alphas:
        raise RuntimeError(f"No checkpoints found under {models_root}. Check the path.")
    return np.mean(np.stack(alphas), axis=0)

# delete stale cache if it exists, then recompute
if alpha_path.exists():
    alpha_path.unlink()
    print("Deleted stale alpha cache.")

alpha_mean = load_alpha_mean_exact(models_root, SEEDS, NUM_LAYERS, LR, L1_W, L2_V, EPOCHS)
np.save(alpha_path, alpha_mean)
print(f"alpha_mean loaded, shape={alpha_mean.shape}, all finite={np.all(np.isfinite(alpha_mean))}")

df = pd.read_csv(per_layer_csv)
langs = sorted(df["test_lang"].dropna().unique().tolist())
if train_lang in langs:
    langs = [train_lang] + [l for l in langs if l != train_lang]

In [ ]:
# -------------------- AUROC --------------------
output_dir = f"../plots/weight_ablation/{DATASET}/{LLM}/plots_auroc_{train_lang}_{probe_type}"
os.makedirs(output_dir, exist_ok=True)
metric_col = "imp_auroc_CAL" if USE_CALIBRATED else "imp_auroc_RAW"

# per-language
for lang in langs:
    dfg = df[df["test_lang"] == lang]
    label = f"few-shot ({lang})" if lang == train_lang else f"zero-shot ({lang})"
    fname = f"few_shot_{lang}" if lang == train_lang else f"zero_shot_{lang}"
    plot_one(dfg, label=label, fname=fname, metric_col=metric_col, metric_name="AUROC",
             ylabel="$\\Delta$AUROC", ylim=(-0.08, 0.08), yticks=np.arange(-0.08, 0.081, 0.02),
             alpha_mean=alpha_mean, output_dir=output_dir, num_layers=NUM_LAYERS)

# pooled (uses full df, not just last dfg)
dfg_all = df[df["test_lang"] != train_lang]
plot_one(dfg_all, label="zero-shot (mean all languages)", fname="zero_shot_all",
         metric_col=metric_col, metric_name="AUROC", ylabel="$\\Delta$AUROC",
         ylim=(-0.08, 0.08), yticks=np.arange(-0.08, 0.081, 0.02),
         alpha_mean=alpha_mean, output_dir=output_dir, num_layers=NUM_LAYERS)


# -------------------- AUPR --------------------
output_dir = f"../plots/weight_ablation/{DATASET}/{LLM}/plots_aupr_{train_lang}_{probe_type}"
os.makedirs(output_dir, exist_ok=True)
metric_col = "imp_aupr_CAL" if USE_CALIBRATED else "imp_aupr_RAW"

for lang in langs:
    dfg = df[df["test_lang"] == lang]
    label = f"few-shot ({lang})" if lang == train_lang else f"zero-shot ({lang})"
    fname = f"few_shot_{lang}" if lang == train_lang else f"zero_shot_{lang}"
    plot_one(dfg, label=label, fname=fname, metric_col=metric_col, metric_name="AUPR",
             ylabel="$\\Delta$AUPR", ylim=(-0.12, 0.12), yticks=np.arange(-0.12, 0.121, 0.03),
             alpha_mean=alpha_mean, output_dir=output_dir, num_layers=NUM_LAYERS)

dfg_all = df[df["test_lang"] != train_lang]
plot_one(dfg_all, label="zero-shot (mean all languages)", fname="zero_shot_all",
         metric_col=metric_col, metric_name="AUPR", ylabel="$\\Delta$AUPR",
         ylim=(-0.12, 0.12), yticks=np.arange(-0.12, 0.121, 0.03),
         alpha_mean=alpha_mean, output_dir=output_dir, num_layers=NUM_LAYERS)


# -------------------- Brier --------------------
output_dir = f"../plots/weight_ablation/{DATASET}/{LLM}/plots_brier_{train_lang}_{probe_type}"
os.makedirs(output_dir, exist_ok=True)
metric_col = "imp_brier_CAL" if USE_CALIBRATED else "imp_brier_RAW"

for lang in langs:
    dfg = df[df["test_lang"] == lang]
    label = f"few-shot ({lang})" if lang == train_lang else f"zero-shot ({lang})"
    fname = f"few_shot_{lang}" if lang == train_lang else f"zero_shot_{lang}"
    plot_one(dfg, label=label, fname=fname, metric_col=metric_col, metric_name="Brier",
             ylabel="$\\Delta$Brier", ylim=(-0.16, 0.16), yticks=np.arange(-0.16, 0.161, 0.04),
             alpha_mean=alpha_mean, output_dir=output_dir, num_layers=NUM_LAYERS)

dfg_all = df[df["test_lang"] != train_lang]
plot_one(dfg_all, label="zero-shot (mean all languages)", fname="zero_shot_all",
         metric_col=metric_col, metric_name="Brier", ylabel="$\\Delta$Brier",
         ylim=(-0.16, 0.16), yticks=np.arange(-0.16, 0.161, 0.04),
         alpha_mean=alpha_mean, output_dir=output_dir, num_layers=NUM_LAYERS)


# -------------------- ECE --------------------
output_dir = f"../plots/weight_ablation/{DATASET}/{LLM}/plots_ece_{train_lang}_{probe_type}"
os.makedirs(output_dir, exist_ok=True)
metric_col = "imp_ece_CAL" if USE_CALIBRATED else "imp_ece_RAW"

for lang in langs:
    dfg = df[df["test_lang"] == lang]
    label = f"few-shot ({lang})" if lang == train_lang else f"zero-shot ({lang})"
    fname = f"few_shot_{lang}" if lang == train_lang else f"zero_shot_{lang}"
    plot_one(dfg, label=label, fname=fname, metric_col=metric_col, metric_name="ECE",
             ylabel="$\\Delta$ECE", ylim=(-0.2, 0.2), yticks=np.arange(-0.2, 0.21, 0.05),
             alpha_mean=alpha_mean, output_dir=output_dir, num_layers=NUM_LAYERS)

dfg_all = df[df["test_lang"] != train_lang]
plot_one(dfg_all, label="zero-shot (mean all languages)", fname="zero_shot_all",
         metric_col=metric_col, metric_name="ECE", ylabel="$\\Delta$ECE",
         ylim=(-0.2, 0.2), yticks=np.arange(-0.2, 0.21, 0.05),
         alpha_mean=alpha_mean, output_dir=output_dir, num_layers=NUM_LAYERS)

## Representation ablation

In [ ]:
# -------------------- CONFIG --------------------
PROBE = "SoftmaxLayerProbe"
L1_W = 0
USE_CALIBRATED = False
SEEDS = [42, 17, 3, 59, 884, 445, 369, 169, 76, 905]
train_lang = "fr"

# DATASET = "mkqa"
DATASET = "global_mmlu"
# LLM   = "llama_3.1_8B"
LLM     = "qwen3_8B"

EXCLUDE_TIME_SENSITIVE = True if DATASET == "mkqa" else False
DATASPLITS   = DATASPLITS_MKQA if DATASET == "mkqa" else DATASPLITS_GMMLU
NUM_LAYERS   = 32 if LLM == "llama_3.1_8B" else 36
HIDDEN_DIM   = 4096
TRAIN_BATCH  = 32
STRATIFIED   = True
K_WINDOW     = 3

RESULTS_ROOT = Path(LOCAL_RESULTS_DIR) / DATASET / LLM

# ---- hyperparams that identify the trained probe ----
# MKQA - LLAMA 3.1 8 B
# QUERY = True
# LR = 0.0008
# EPOCHS = 139 
# L2_V = 0.286102950514828

# QUERY = False
# LR = 0.0005
# EPOCHS = 210
# L2_V = 0.27276092363216636

# ---- MKQA - QWEN 3 8B
# QUERY = True
# LR = 0.0009
# EPOCHS = 98
# L2_V = 0.29527869618547636

# QUERY = False
# LR = 0.0009
# EPOCHS = 111
# L2_V = 0.22252863623823427

# ---- GLOBAL MMLU - LLAMA 3.1 8B
# QUERY = True
# LR = 0.0011
# EPOCHS = 94 
# L2_V = 0.28258845606106203

# QUERY = False
# LR = 0.0020
# EPOCHS = 60
# L2_V = 0.2557367492257226

# ---- GLOBAL MMLU - QWEN 3 8B
# QUERY = True
# LR = 0.001
# EPOCHS = 103
# L2_V = 0.2686538578969079

QUERY = False
LR = 0.001
EPOCHS = 165
L2_V = 0.2115332075762067

In [ ]:
# -------------------- PATH BUILDING --------------------
probe_type = "query" if QUERY else "answer"
if QUERY:
    probe_folder = "probe_query_without_time_sensitive" if EXCLUDE_TIME_SENSITIVE else "probe_query"
else:
    probe_folder = "probe_answer_without_time_sensitive" if EXCLUDE_TIME_SENSITIVE else "probe_answer"

per_layer_csv = (
    Path(LOCAL_RESULTS_DIR)
    / f"{DATASET}/{LLM}/{probe_folder}"
    / f"SoftmaxLayerProbe_layer_scalers/train_{train_lang}"
    / f"lr_{LR}_l1_{L1_W}_l2_{L2_V}_epochs_{EPOCHS}_representation_k{K_WINDOW}_per_layer_multiseed.csv"
)
models_root = (
    Path(LOCAL_RESULTS_DIR)
    / f"{DATASET}/{LLM}/{probe_folder}"
    / f"SoftmaxLayerProbe_layer_scalers/train_{train_lang}"
)

In [ ]:
# -------------------- α: compute once from EXACT seed paths --------------------
alpha_path = models_root / f"alpha_mean_lr_{LR}_l1_{L1_W}_l2_{L2_V}_epochs_{EPOCHS}.npy"

def load_alpha_mean_exact(
    models_root: Path, seeds, num_layers: int,
    lr: float, l1_w: float, l2_v: float, epochs: int
) -> np.ndarray:
    alphas = []
    missing = []
    for seed in seeds:
        ckpt = (
            models_root
            / f"batch_32_lr_{lr}_l1_{l1_w}_l2_{l2_v}_epochs_{epochs}_seed_{seed}"  # FIX: added batch_32_ prefix
            / "probe_state.pt"
        )
        if not ckpt.exists():
            missing.append(seed)
            continue
        st = torch.load(ckpt, map_location="cpu")
        w = st["w"].reshape(-1)
        if w.numel() != num_layers:
            raise ValueError(f"[seed {seed}] expected {num_layers} layers, got {w.numel()}")
        alphas.append(F.softmax(w, dim=-1).numpy())
    if missing:
        print(f"[warn] missing checkpoints for seeds: {missing}")
    if not alphas:
        raise RuntimeError(f"No checkpoints found under {models_root}. Check the path.")
    return np.mean(np.stack(alphas), axis=0)

# delete stale cache if it exists, then recompute
if alpha_path.exists():
    alpha_path.unlink()
    print("Deleted stale alpha cache.")

alpha_mean = load_alpha_mean_exact(models_root, SEEDS, NUM_LAYERS, LR, L1_W, L2_V, EPOCHS)
np.save(alpha_path, alpha_mean)
print(f"alpha_mean loaded, shape={alpha_mean.shape}, all finite={np.all(np.isfinite(alpha_mean))}")

df = pd.read_csv(per_layer_csv)
langs = sorted(df["test_lang"].dropna().unique().tolist())
if train_lang in langs:
    langs = [train_lang] + [l for l in langs if l != train_lang]

In [ ]:
# -------------------- AUROC --------------------
output_dir = f"../plots/representation_ablation/{DATASET}/{LLM}/plots_auroc_{train_lang}_{probe_type}"
os.makedirs(output_dir, exist_ok=True)
metric_col = "imp_auroc_CAL" if USE_CALIBRATED else "imp_auroc_RAW"

# per-language
for lang in langs:
    dfg = df[df["test_lang"] == lang]
    label = f"few-shot ({lang})" if lang == train_lang else f"zero-shot ({lang})"
    fname = f"few_shot_{lang}" if lang == train_lang else f"zero_shot_{lang}"
    plot_one(dfg, label=label, fname=fname, metric_col=metric_col, metric_name="AUROC",
             ylabel="$\\Delta$AUROC", ylim=(-0.08, 0.08), yticks=np.arange(-0.08, 0.081, 0.02),
             alpha_mean=alpha_mean, output_dir=output_dir, num_layers=NUM_LAYERS)

# pooled (uses full df, not just last dfg)
dfg_all = df[df["test_lang"] != train_lang]
plot_one(dfg_all, label="zero-shot (mean all languages)", fname="zero_shot_all",
         metric_col=metric_col, metric_name="AUROC", ylabel="$\\Delta$AUROC",
         ylim=(-0.08, 0.08), yticks=np.arange(-0.08, 0.081, 0.02),
         alpha_mean=alpha_mean, output_dir=output_dir, num_layers=NUM_LAYERS)


# -------------------- AUPR --------------------
output_dir = f"../plots/representation_ablation/{DATASET}/{LLM}/plots_aupr_{train_lang}_{probe_type}"
os.makedirs(output_dir, exist_ok=True)
metric_col = "imp_aupr_CAL" if USE_CALIBRATED else "imp_aupr_RAW"

for lang in langs:
    dfg = df[df["test_lang"] == lang]
    label = f"few-shot ({lang})" if lang == train_lang else f"zero-shot ({lang})"
    fname = f"few_shot_{lang}" if lang == train_lang else f"zero_shot_{lang}"
    plot_one(dfg, label=label, fname=fname, metric_col=metric_col, metric_name="AUPR",
             ylabel="$\\Delta$AUPR", ylim=(-0.12, 0.12), yticks=np.arange(-0.12, 0.121, 0.03),
             alpha_mean=alpha_mean, output_dir=output_dir, num_layers=NUM_LAYERS)

dfg_all = df[df["test_lang"] != train_lang]
plot_one(dfg_all, label="zero-shot (mean all languages)", fname="zero_shot_all",
         metric_col=metric_col, metric_name="AUPR", ylabel="$\\Delta$AUPR",
         ylim=(-0.12, 0.12), yticks=np.arange(-0.12, 0.121, 0.03),
         alpha_mean=alpha_mean, output_dir=output_dir, num_layers=NUM_LAYERS)


# -------------------- Brier --------------------
output_dir = f"../plots/representation_ablation/{DATASET}/{LLM}/plots_brier_{train_lang}_{probe_type}"
os.makedirs(output_dir, exist_ok=True)
metric_col = "imp_brier_CAL" if USE_CALIBRATED else "imp_brier_RAW"

for lang in langs:
    dfg = df[df["test_lang"] == lang]
    label = f"few-shot ({lang})" if lang == train_lang else f"zero-shot ({lang})"
    fname = f"few_shot_{lang}" if lang == train_lang else f"zero_shot_{lang}"
    plot_one(dfg, label=label, fname=fname, metric_col=metric_col, metric_name="Brier",
             ylabel="$\\Delta$Brier", ylim=(-0.16, 0.16), yticks=np.arange(-0.16, 0.161, 0.04),
             alpha_mean=alpha_mean, output_dir=output_dir, num_layers=NUM_LAYERS)

dfg_all = df[df["test_lang"] != train_lang]
plot_one(dfg_all, label="zero-shot (mean all languages)", fname="zero_shot_all",
         metric_col=metric_col, metric_name="Brier", ylabel="$\\Delta$Brier",
         ylim=(-0.16, 0.16), yticks=np.arange(-0.16, 0.161, 0.04),
         alpha_mean=alpha_mean, output_dir=output_dir, num_layers=NUM_LAYERS)


# -------------------- ECE --------------------
output_dir = f"../plots/representation_ablation/{DATASET}/{LLM}/plots_ece_{train_lang}_{probe_type}"
os.makedirs(output_dir, exist_ok=True)
metric_col = "imp_ece_CAL" if USE_CALIBRATED else "imp_ece_RAW"

for lang in langs:
    dfg = df[df["test_lang"] == lang]
    label = f"few-shot ({lang})" if lang == train_lang else f"zero-shot ({lang})"
    fname = f"few_shot_{lang}" if lang == train_lang else f"zero_shot_{lang}"
    plot_one(dfg, label=label, fname=fname, metric_col=metric_col, metric_name="ECE",
             ylabel="$\\Delta$ECE", ylim=(-0.2, 0.2), yticks=np.arange(-0.2, 0.21, 0.05),
             alpha_mean=alpha_mean, output_dir=output_dir, num_layers=NUM_LAYERS)

dfg_all = df[df["test_lang"] != train_lang]
plot_one(dfg_all, label="zero-shot (mean all languages)", fname="zero_shot_all",
         metric_col=metric_col, metric_name="ECE", ylabel="$\\Delta$ECE",
         ylim=(-0.2, 0.2), yticks=np.arange(-0.2, 0.21, 0.05),
         alpha_mean=alpha_mean, output_dir=output_dir, num_layers=NUM_LAYERS)

## Spearman's r between probe and few/zero-shot

In [ ]:
METRIC_CONFIGS = [
    dict(name="auroc", lim=0.08,  step=0.02),
    dict(name="aupr",  lim=0.12,  step=0.03),
    dict(name="brier", lim=0.16,  step=0.04),
    dict(name="ece",   lim=0.20,  step=0.05),
]

# -------------------- CONFIG --------------------
PROBE = "SoftmaxLayerProbe"
L1_W = 0
USE_CALIBRATED = False
SEEDS = [42, 17, 3, 59, 884, 445, 369, 169, 76, 905]
train_lang = "fr"

# DATASET = "mkqa"
DATASET = "global_mmlu"
# LLM   = "llama_3.1_8B"
LLM     = "qwen3_8B"

EXCLUDE_TIME_SENSITIVE = True if DATASET == "mkqa" else False
DATASPLITS   = DATASPLITS_MKQA if DATASET == "mkqa" else DATASPLITS_GMMLU
NUM_LAYERS   = 32 if LLM == "llama_3.1_8B" else 36
HIDDEN_DIM   = 4096
TRAIN_BATCH  = 32
STRATIFIED   = True
K_WINDOW     = 3

RESULTS_ROOT = Path(LOCAL_RESULTS_DIR) / DATASET / LLM

# ---- hyperparams that identify the trained probe ----
# MKQA - LLAMA 3.1 8 B
# QUERY = True
# LR = 0.0008
# EPOCHS = 139 
# L2_V = 0.286102950514828

# QUERY = False
# LR = 0.0005
# EPOCHS = 210
# L2_V = 0.27276092363216636

# ---- MKQA - QWEN 3 8B
# QUERY = True
# LR = 0.0009
# EPOCHS = 98
# L2_V = 0.29527869618547636

# QUERY = False
# LR = 0.0009
# EPOCHS = 111
# L2_V = 0.22252863623823427

# ---- GLOBAL MMLU - LLAMA 3.1 8B
# QUERY = True
# LR = 0.0011
# EPOCHS = 94 
# L2_V = 0.28258845606106203

# QUERY = False
# LR = 0.0020
# EPOCHS = 60
# L2_V = 0.2557367492257226

# ---- GLOBAL MMLU - QWEN 3 8B
# QUERY = True
# LR = 0.001
# EPOCHS = 103
# L2_V = 0.2686538578969079

QUERY = False
LR = 0.001
EPOCHS = 165
L2_V = 0.2115332075762067

In [ ]:
import pandas as pd
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

# -------------------- PATHS (mirrors per_layer_csv construction) --------------------
if QUERY == True:
    probe_folder = "probe_query_without_time_sensitive" if EXCLUDE_TIME_SENSITIVE else "probe_query"
else: 
    probe_folder = "probe_answer_without_time_sensitive" if EXCLUDE_TIME_SENSITIVE else "probe_answer"
suffix     = "CAL" if USE_CALIBRATED else "RAW"
base_stem  = f"lr_{LR}_l1_{L1_W}_l2_{L2_V}_epochs_{EPOCHS}"
probe_dir  = Path(LOCAL_RESULTS_DIR) / DATASET / LLM / probe_folder / f"SoftmaxLayerProbe_layer_scalers/train_{train_lang}"

probe_file = probe_dir / f"{base_stem}_multiseed_eval_layer_importance_summary_pure.csv"
repr_file  = probe_dir / f"{base_stem}_representation_k{K_WINDOW}_per_layer_multiseed_summary.csv"

# -------------------- LOAD --------------------
probe_df = pd.read_csv(probe_file)
repr_df  = pd.read_csv(repr_file)

# -------------------- BUILD LAYER TABLE --------------------
# Probe: mean importance per layer (used as the reference signal)
probe_baseline = (
    probe_df[["layer", "importance_mean"]]
    .rename(columns={"importance_mean": "probe_importance"})
)

# Column names in repr_df, derived from METRIC_CONFIGS + USE_CALIBRATED
metric_cols = {m["name"]: f"imp_{m['name']}_{suffix}_mean" for m in METRIC_CONFIGS}

# Few-shot: train language only
fs = (
    repr_df.loc[repr_df["test_lang"] == train_lang, ["layer"] + list(metric_cols.values())]
    .copy()
    .rename(columns={col: f"FS_{name}" for name, col in metric_cols.items()})
)

# Zero-shot: mean over all non-train languages per layer
zs = (
    repr_df.loc[repr_df["test_lang"] != train_lang, ["layer"] + list(metric_cols.values())]
    .groupby("layer", as_index=False)
    .mean(numeric_only=True)
    .rename(columns={col: f"ZS_{name}" for name, col in metric_cols.items()})
)

layer_values = probe_baseline.merge(fs, on="layer").merge(zs, on="layer")

# -------------------- SPEARMAN CORRELATIONS --------------------
rows = []
for setting in ("FS", "ZS"):
    for name in metric_cols:
        rho, p = spearmanr(
            layer_values["probe_importance"],
            layer_values[f"{setting}_{name}"],
            nan_policy="omit",
        )
        rows.append({"setting": setting, "metric": name,
                     "n": len(layer_values), "spearman_rho": rho, "spearman_p": p})

results = pd.DataFrame(rows)
results["spearman_p_BH"] = multipletests(results["spearman_p"], method="fdr_bh")[1]

# -------------------- SAVE --------------------
out_dir = probe_dir  # keep outputs next to inputs
layer_values.to_csv(out_dir / f"{base_stem}_correlation_layer_values.csv",      index=False)
results.to_csv(     out_dir / f"{base_stem}_spearman_correlation_results.csv",   index=False)
print(results.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -------------------- PIVOT --------------------
# results has columns: setting, metric, spearman_rho
pivot = results.pivot(index="metric", columns="setting", values="spearman_rho")
# Keep the same metric order as METRIC_CONFIGS
metric_order = [m["name"] for m in METRIC_CONFIGS]
pivot = pivot.reindex(metric_order)

# -------------------- PLOT --------------------
fig, ax = plt.subplots(figsize=(7, 4))

n_metrics = len(pivot)
x = np.arange(n_metrics)
bar_width = 0.35

colors = {"FS": "#1F4E79", "ZS": "#C55A11"}   # dark blue / burnt orange (matches screenshot)

for i, setting in enumerate(["FS", "ZS"]):
    offset = (i - 0.5) * bar_width
    bars = ax.bar(x + offset, pivot[setting], width=bar_width,
                  label=setting, color=colors[setting])

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(pivot.index)
ax.set_ylabel("Spearman $\\rho$")
ax.set_title(f"Layer-importance correlation — {DATASET} / {LLM}")
ax.set_ylim(-0.5, 1.05)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.grid(True, linewidth=0.4, color="lightgrey")
ax.set_axisbelow(True)

plt.tight_layout()

# -------------------- SAVE --------------------
out_path = probe_dir / f"{base_stem}_spearman_bar.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out_path}")